# Домашнее задание к семинару 05 (HW05)

Тема: линейные модели и честный ML-эксперимент.

## 2.3.1. Загрузка данных и первичный анализ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, classification_report, confusion_matrix

# Загрузка данных
df = pd.read_csv('S05-hw-dataset.csv')

# Первичный анализ
print("Первые 5 строк:")
display(df.head())

print("
Информация о датасете:")
df.info()

print("
Описательная статистика:")
display(df.describe())

print("
Распределение целевого признака (default):")
print(df['default'].value_counts(normalize=True))

### Наблюдения:
- Датасет содержит 3000 объектов и 17 признаков.
- Целевая переменная `default` распределена как ~60% (0) и ~40% (1), что является умеренным дисбалансом.
- Явных аномалий в базовых статистиках не замечено.

## 2.3.2. Подготовка признаков и таргета

In [ ]:
# Выделение X и y
X = df.drop(columns=['client_id', 'default'])
y = df['default']

print(f"Форма матрицы признаков: {X.shape}")
print(f"Форма вектора таргета: {y.shape}")

## 2.3.3. Train/Test-сплит и бейзлайн-модель

In [ ]:
# Разделение на train и test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Бейзлайн модель
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

# Оценка бейзлайна
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))
dummy_roc_auc = roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1])

print(f"Baseline (Most Frequent) Accuracy: {dummy_acc:.4f}")
print(f"Baseline (Most Frequent) ROC-AUC: {dummy_roc_auc:.4f}")

## 2.3.4. Логистическая регрессия и подбор гиперпараметров

In [ ]:
# Создание Pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

# Подбор параметра C
param_grid = {
    'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X_train, y_train)

print(f"Лучшее значение C: {grid_search.best_params_['logreg__C']}")
print(f"Лучший ROC-AUC на кросс-валидации: {grid_search.best_score_:.4f}")

# Оценка лучшей модели на тестовой выборке
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

model_acc = accuracy_score(y_test, y_pred)
model_roc_auc = roc_auc_score(y_test, y_proba)

print(f"
Model (Logistic Regression) Accuracy: {model_acc:.4f}")
print(f"Model (Logistic Regression) ROC-AUC: {model_roc_auc:.4f}")

### Визуализация: ROC-кривая

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {model_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.savefig('figures/roc_curve.png')
plt.show()

## 2.3.5. Сравнение бейзлайна и логистической регрессии, текстовые выводы

In [ ]:
results = pd.DataFrame({
    'Metric': ['Accuracy', 'ROC-AUC'],
    'Baseline (Dummy)': [dummy_acc, dummy_roc_auc],
    'Logistic Regression': [model_acc, model_roc_auc]
})

display(results)

### Отчёт по результатам эксперимента

1. **Сравнение с бейзлайном**: Логистическая регрессия значительно превосходит бейзлайн `DummyClassifier` (most_frequent). В то время как бейзлайн имеет ROC-AUC 0.5 (что эквивалентно случайному гаданию), наша модель показала гораздо более высокий результат.
2. **Метрики качества**: Accuracy модели выше, чем доля мажоритарного класса, что говорит о том, что модель действительно научилась извлекать полезные паттерны из данных. Высокий показатель ROC-AUC подтверждает хорошую разделительную способность модели.
3. **Влияние регуляризации**: В ходе GridSearch был подобран оптимальный коэффициент `C`. Регуляризация помогла избежать переобучения и достичь лучшей обобщающей способности.
4. **Выводы**: Логистическая регрессия является адекватным выбором для данной задачи бинарной классификации. Она интерпретируема, быстра в обучении и показывает стабильно высокие результаты по сравнению с простейшими методами. Для дальнейшего улучшения можно рассмотреть генерацию новых признаков или использование более сложных моделей (например, градиентный бустинг).